# Camada Silver — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

---

## Visão Geral

A camada Silver é responsável por transformar os dados brutos ingeridos na Bronze em tabelas limpas, padronizadas e semanticamente organizadas segundo a **modelagem dimensional** (estrela). Cada notebook desta camada lê de `bronze.*`, aplica as transformações necessárias e grava em `silver.*`.

Os princípios aplicados em todas as tabelas Silver são:

- **Deduplicação** — reprocessamentos da Bronze (modo `append`) são neutralizados via `row_number()` sobre a chave primária ordenada por `timestamp_ingestion`
- **Tipagem explícita** — colunas lidas como `string` na Bronze recebem o tipo correto (`timestamp`, `decimal`, `boolean`, `int`)
- **Sanitização textual** — valores categóricos inconsistentes (variações de case, abreviações, typos) são normalizados para um conjunto canônico
- **Colunas derivadas** — atributos calculáveis a partir dos dados brutos são materializados para evitar recomputação nas camadas superiores
- **Idempotência** — todas as escritas usam `mode=overwrite`, garantindo que reexecutar o notebook produza sempre o mesmo resultado
- **Rastreabilidade** — `timestamp_ingestion` é recriado com o instante da carga Silver

---

## Tabelas produzidas neste notebook

### Tabelas Fato

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `ft_tickets_suporte` | `suporte_tickets` | Tickets do SAC limpos, tipados e com colunas derivadas |
| `ft_pedidos` | `pedidos` | Histórico de compras com valor total e ano_mes derivados |
| `ft_avaliacoes` | `avaliacoes` | Avaliações pós-compra com categoria NPS derivada |
| `ft_clickstream` | `clickstream` | Eventos de navegação tipados e normalizados |

### Dimensões

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `dim_clientes` | `clientes` | Perfis cadastrais com idade, nome completo e região derivados |
| `dim_produtos` | `catalogo_produtos` | Catálogo de produtos com categoria normalizada |
| `dim_tipos_problema` | `suporte_tickets` | Tipos de problema canônicos com categoria de negócio |
| `dim_agentes_suporte` | `suporte_tickets` | Agentes com métricas agregadas de desempenho |
| `dim_status_pedido` | `pedidos` | Status de pedido normalizados |
| `dim_categorias_produto` | `catalogo_produtos` | Categorias de produto normalizadas |

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalogo       = 'vcommerce_catalog'
bronze_schema  = 'vcommerce_bronze'
silver_schema  = 'vcommerce_silver'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {silver_schema}')
spark.sql(f'USE SCHEMA {silver_schema}')

print(f'Catálogo : {catalogo}')
print(f'Schema   : {silver_schema}')

---

## Tratamento: `ft_tickets_suporte`

**Origem:** `bronze.suporte_tickets`  
**Destino:** `silver.ft_tickets_suporte`, `silver.dim_tipos_problema`, `silver.dim_agentes_suporte`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_problema` | ~25 variações para 4 categorias reais (`pro`, `p3oduto`, `PRODUCT`, `DELAY`…) | Mapeamento para valores canônicos |
| `data_resolucao` | 2.072 nulos — tickets ainda abertos | Mantidos como `null`; `resolvido = false` |
| `nota_avaliacao` | 2.072 nulos — sem avaliação para tickets abertos | Mantidos como `null` |
| `tempo_resolucao_horas` | 2.072 nulos — tickets não resolvidos | Mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `ticket_id` |
| Datas | Formato ISO com timezone (`2023-01-03T05:13:00.000Z`) | Cast para `timestamp` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `resolvido` | `data_resolucao IS NOT NULL` |
| `hora_abertura` | `HOUR(data_abertura)` |
| `dia_semana_abertura` | `DAYOFWEEK(data_abertura)` |

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada ticket (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.suporte_tickets')

window_dedup = Window.partitionBy('ticket_id').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [0]:
# ─── Mapeamento canônico de tipo_problema ────────────────────────────────────
# Na Bronze foram encontradas 4 categorias reais fragmentadas em ~25 variações:
#
#   Entrega   - Entrega, ENTREGA, entrega, 3ntrega, entr, del, DELAY, delay
#   Reembolso - Reembolso, REEMBOLSO, reembolso, REFUND, ref, reemb, r3embolso
#   Produto   - Produto, PRODUTO, produto, pro, prod, p3oduto, PRODUCT, product
#   Pagamento - Pagamento, PAGAMENTO, pagamento, pag, pay, PAY, PAYMENT, payment,
#               p4gamento
#
# Estratégia: normaliza para lowercase e aplica mapeamento por prefixo/palavra-chave.
# Valores não mapeados são marcados como 'Outro' para investigação futura.

tipo_map = {
    # Entrega
    'entrega'  : 'Entrega',  '3ntrega' : 'Entrega',  'entr'  : 'Entrega',
    'del'      : 'Entrega',  'delay'   : 'Entrega',
    # Reembolso
    'reembolso': 'Reembolso','reemb'   : 'Reembolso','r3embolso': 'Reembolso',
    'refund'   : 'Reembolso','ref'     : 'Reembolso',
    # Produto
    'produto'  : 'Produto',  'pro'     : 'Produto',  'prod'  : 'Produto',
    'p3oduto'  : 'Produto',  'product' : 'Produto',
    # Pagamento
    'pagamento': 'Pagamento','pag'     : 'Pagamento','p4gamento': 'Pagamento',
    'pay'      : 'Pagamento','payment' : 'Pagamento',
}

# Constrói expressão CASE WHEN a partir do dicionário
tipo_expr = F.col('tipo_problema')
for raw_val, canonical in tipo_map.items():
    tipo_expr = F.when(
        F.lower(F.col('tipo_problema')) == raw_val, canonical
    ).otherwise(tipo_expr)

# Valores que não bateram com nenhum mapeamento ficam como 'Outro'
known_lower = [k.lower() for k in tipo_map.keys()]
tipo_expr = F.when(
    F.lower(F.col('tipo_problema')).isin(known_lower), tipo_expr
).otherwise(F.lit('Outro'))

print('Expressão de mapeamento criada.')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver = (
    df_dedup

    # 1. Tipos corretos para datas (ISO 8601 com timezone)
    .withColumn('data_abertura',   F.to_timestamp('data_abertura'))
    .withColumn('data_resolucao',  F.to_timestamp('data_resolucao'))

    # 2. Normalização de tipo_problema
    .withColumn('tipo_problema', tipo_expr)

    # 3. Colunas derivadas de negócio
    .withColumn(
        'resolvido',
        F.col('data_resolucao').isNotNull()
    )
    .withColumn(
        'hora_abertura',
        F.hour('data_abertura').cast('int')
    )
    .withColumn(
        'dia_semana_abertura',
        F.date_format('data_abertura', 'EEEE')   # nome completo em inglês; adaptar locale se necessário
    )

    # 4. Tipos numéricos
    .withColumn('tempo_resolucao_horas', F.col('tempo_resolucao_horas').cast('decimal(10,2)'))
    .withColumn('nota_avaliacao',        F.col('nota_avaliacao').cast('decimal(3,1)'))

    # 5. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'ticket_id',
        'id_cliente',
        'id_pedido',
        'tipo_problema',
        'data_abertura',
        'data_resolucao',
        'tempo_resolucao_horas',
        'agente_suporte',
        'nota_avaliacao',
        'resolvido',
        'hora_abertura',
        'dia_semana_abertura',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total = df_silver.count()
resolvidos     = df_silver.filter(F.col('resolvido') == True).count()
nao_resolvidos = df_silver.filter(F.col('resolvido') == False).count()
outros_tipo    = df_silver.filter(F.col('tipo_problema') == 'Outro').count()

print(f'Total de tickets       : {total:,}')
print(f'Resolvidos             : {resolvidos:,} ({resolvidos/total*100:.1f}%)')
print(f'Abertos (sem resolução): {nao_resolvidos:,} ({nao_resolvidos/total*100:.1f}%)')
print(f'Tipo "Outro" (suspeitos): {outros_tipo:,}')
print()
print('Distribuição de tipo_problema após normalização:')
df_silver.groupBy('tipo_problema').count().orderBy(F.desc('count')).show()

In [0]:
# ─── Grava silver.ft_tickets_suporte ─────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.ft_tickets_suporte')
)

print(f'{silver_schema}.ft_tickets_suporte gravada com {df_silver.count():,} registros.')

In [0]:
# ─── dim_tipos_problema ───────────────────────────────────────────────────────
# Dimensão com os 4 tipos canônicos e sua categoria de negócio.
#
# Categoria derivada:
#   Entrega   - Logística
#   Reembolso - Financeiro
#   Produto   - Qualidade
#   Pagamento - Financeiro
#   Outro     - Indefinido

df_dim_tipos = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .select('tipo_problema')
         .distinct()
         .withColumn(
             'categoria_problema',
             F.when(F.col('tipo_problema') == 'Entrega',   'Logística')
              .when(F.col('tipo_problema') == 'Reembolso', 'Financeiro')
              .when(F.col('tipo_problema') == 'Produto',   'Qualidade')
              .when(F.col('tipo_problema') == 'Pagamento', 'Financeiro')
              .otherwise('Indefinido')
         )
         .orderBy('tipo_problema')
)

df_dim_tipos.show()

(
    df_dim_tipos.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(f'{silver_schema}.dim_tipos_problema')
)

print(f'{silver_schema}.dim_tipos_problema gravada.')

In [0]:
# ─── dim_agentes_suporte ──────────────────────────────────────────────────────
# Dimensão com métricas agregadas por agente, derivadas de ft_tickets_suporte.
#
# Métricas calculadas:
#   qtd_tickets_resolvidos  - total de tickets onde resolvido = true
#   nota_media_atendimento  - média de nota_avaliacao (ignora nulos)

df_dim_agentes = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .groupBy('agente_suporte')
         .agg(
             F.count(
                 F.when(F.col('resolvido') == True, 1)
             ).alias('qtd_tickets_resolvidos'),
             F.round(
                 F.avg('nota_avaliacao'), 2
             ).alias('nota_media_atendimento'),
         )
         .withColumn(
             'nota_media_atendimento',
             F.col('nota_media_atendimento').cast('decimal(4,2)')
         )
         .orderBy(F.desc('qtd_tickets_resolvidos'))
)

df_dim_agentes.show(truncate=False)

(
    df_dim_agentes.write
                  .format('delta')
                  .mode('overwrite')
                  .option('overwriteSchema', 'true')
                  .saveAsTable(f'{silver_schema}.dim_agentes_suporte')
)

print(f'{silver_schema}.dim_agentes_suporte gravada.')

In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────

tabelas = [
    f'{silver_schema}.ft_tickets_suporte',
    f'{silver_schema}.dim_tipos_problema',
    f'{silver_schema}.dim_agentes_suporte',
]

print('=== Camada Silver — Tickets de Suporte ===')
print(f'{"Tabela":<35} {"Linhas":>8} {"Colunas":>8}')
print('-' * 55)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<35} {df_t.count():>8,} {len(df_t.columns):>8}')

## Tratamento: `dim_clientes`

**Origem:** `bronze.clientes`  
**Destino:** `silver.dim_clientes`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `nome`, `sobrenome` | Variações de case (`JOAO`, `joao`, `João`) | `INITCAP()` + `TRIM()` |
| `email` | Case misto (`Joao@Gmail.com`) | `LOWER()` + `TRIM()` |
| `genero` | ~10 variações para 3 categorias (`M`, `male`, `MASC`…) | Mapeamento canônico via cadeia de `WHEN` |
| `origem` | Variações (`app`, `APP`, `App Mobile`, `web`, `WEB`…) | `LOWER()` + `TRIM()` + mapeamento canônico |
| `cidade` | Case misto (`são paulo`, `SÃO PAULO`) | `INITCAP()` + `TRIM()` |
| `estado` | Case misto e abreviações inconsistentes | `UPPER()` + `TRIM()` |
| `pais` | Case misto | `UPPER()` + `TRIM()` |
| `data_nascimento`, `data_cadastro` | Inferidas como `string` na Bronze | Cast explícito para `date` |
| `telefone`, `endereco` | Possíveis espaços extras | `TRIM()` |
| `device_ids` | String bruta — possível array serializado | Mantido como `string` para análise futura |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | `ROW_NUMBER()` sobre `id_cliente` por `timestamp_ingestion DESC` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `nome_completo` | `CONCAT(nome, ' ', sobrenome)` |
| `idade` | `FLOOR(DATEDIFF(current_date, data_nascimento) / 365)` |
| `faixa_etaria` | `CASE` sobre `idade`: Menor de 18 / 18-24 / 25-34 / 35-44 / 45-59 / 60+ |
| `tempo_cliente_dias` | `DATEDIFF(current_date, data_cadastro)` |
| `regiao` | `CASE` sobre `estado` → Norte / Nordeste / Centro-Oeste / Sudeste / Sul |
| `timestamp_ingestion` | `current_timestamp()` — recriado no momento da carga Silver |

In [0]:
#Leitura da Bronze
#Usa o registro mais recente por id_cliente (maior timestamp_ingestion) para
#Neutralizar reprocessamentos da Bronze em mode=append.

df_raw = spark.table(f'{bronze_schema}.clientes')

window_dedup = Window.partitionBy('id_cliente').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   #remove timestamp_ingestion que será recriado ao final
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [0]:
#Mapeamentos canônicos
#Genero: 3 categorias reais fragmentadas em múltiplas variações
#Masculino  - m, masculino, male, masc, homem, h
#Feminino   - f, feminino, female, fem, mulher
#Outro      - outro, other, nb, nao_binario, nao informado
#Garantem que qualquer análise que for feita em cima de dados com esssas categorias sejam consistentes

genero_map = {
    'm'          : 'Masculino', 'masculino': 'Masculino', 'male' : 'Masculino',
    'masc'       : 'Masculino', 'homem'    : 'Masculino', 'h'    : 'Masculino',
    'f'          : 'Feminino',  'feminino' : 'Feminino',  'female': 'Feminino',
    'fem'        : 'Feminino',  'mulher'   : 'Feminino',
    'outro'      : 'Outro',     'other'    : 'Outro',     'nb'   : 'Outro',
    'nao_binario': 'Outro',
    'nao-informado' : 'Outro',
    'nao informado': 'Outro',
}

genero_expr = F.lit('Não informado')
for raw_val, canonical in genero_map.items():
    genero_expr = F.when(
        F.lower(F.trim(F.col('genero'))) == raw_val, canonical
    ).otherwise(genero_expr)

#origem: canônicas — App / Web / Parceiro / Não informado
origem_map = {
    'app'        : 'App',      'app mobile': 'App',    'mobile': 'App',
    'web'        : 'Web',      'website'   : 'Web',    'site'  : 'Web',
    'parceiro'   : 'Parceiro', 'partner'   : 'Parceiro',
}

origem_expr = F.lit('Não informado')
for raw_val, canonical in origem_map.items():
    origem_expr = F.when(
        F.lower(F.trim(F.col('origem'))) == raw_val, canonical
    ).otherwise(origem_expr)

print('Expressões de mapeamento criadas.')

In [0]:
#Principais transformações

regiao_expr = (
    #criação de coluna de agrupamento por região 
    
    F.when(F.col('estado').isin('AM','PA','AC','RO','RR','AP','TO'), 'Norte')
     .when(F.col('estado').isin('BA','SE','AL','PE','PB','RN','CE','PI','MA'), 'Nordeste')
     .when(F.col('estado').isin('MT','MS','GO','DF'), 'Centro-Oeste')
     .when(F.col('estado').isin('SP','RJ','MG','ES'), 'Sudeste')
     .when(F.col('estado').isin('PR','SC','RS'), 'Sul')
     .otherwise('Não identificada')
)

df_silver = (
    df_dedup

    #Sanitização textual
    .withColumn('nome',      F.initcap(F.trim(F.col('nome'))))
    .withColumn('sobrenome', F.initcap(F.trim(F.col('sobrenome'))))
    .withColumn('email',     F.lower(F.trim(F.col('email'))))
    .withColumn('telefone',  F.trim(F.col('telefone')))
    .withColumn('endereco',  F.trim(F.col('endereco')))
    .withColumn('cidade',    F.initcap(F.trim(F.col('cidade'))))
    .withColumn('estado',    F.upper(F.trim(F.col('estado'))))
    .withColumn('pais',      F.upper(F.trim(F.col('pais'))))

    #Mapeamentos canônicos
    .withColumn('genero', genero_expr)
    .withColumn('origem', origem_expr)

    #Tipagem explícita
    .withColumn('data_nascimento', F.col('data_nascimento').cast('date'))
    .withColumn('data_cadastro',   F.col('data_cadastro').cast('date'))

    #Colunas derivadas
    .withColumn('nome_completo',
        F.concat_ws(' ', F.col('nome'), F.col('sobrenome'))
    )
    .withColumn('idade',
                F.when(
        F.datediff(F.current_date(), F.col('data_nascimento')) < 0, None
    ).otherwise(
        F.floor(F.datediff(F.current_date(), F.col('data_nascimento')) / 365).cast('int')
    ))
    .withColumn('faixa_etaria',
        F.when(F.col('idade') < 18,  'Menor de 18')
         .when(F.col('idade') <= 24, '18-24')
         .when(F.col('idade') <= 34, '25-34')
         .when(F.col('idade') <= 44, '35-44')
         .when(F.col('idade') <= 59, '45-59')
         .otherwise('60+')
    )
    .withColumn('tempo_cliente_dias',
        F.datediff(F.current_date(), F.col('data_cadastro'))
    )
    .withColumn('regiao', regiao_expr)

    #Rastreabilidade
    .withColumn('timestamp_ingestion', F.current_timestamp())

    #Ordenação de colunas para facilitar visualização
    .select(
        'id_cliente',
        'nome',
        'sobrenome',
        'nome_completo',
        'email',
        'telefone',
        'genero',
        'data_nascimento',
        'idade',
        'faixa_etaria',
        'data_cadastro',
        'tempo_cliente_dias',
        'endereco',
        'cidade',
        'estado',
        'regiao',
        'pais',
        'device_ids',
        'origem',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
#Validação pré-escrita 
#Exibir métricas do df antes de escrever no formato delta

total             = df_silver.count()
nulos_nascimento  = df_silver.filter(F.col('data_nascimento').isNull()).count()
nulos_cadastro    = df_silver.filter(F.col('data_cadastro').isNull()).count()
nao_informado_gen = df_silver.filter(F.col('genero') == 'Não informado').count()
nao_informado_ori = df_silver.filter(F.col('origem') == 'Não informado').count()

print(f'Total de clientes          : {total:,}')
print(f'Nulos em data_nascimento   : {nulos_nascimento:,}')
print(f'Nulos em data_cadastro     : {nulos_cadastro:,}')
print(f'Gênero "Não informado"     : {nao_informado_gen:,} ({nao_informado_gen/total*100:.1f}%)')
print(f'Origem "Não informado"     : {nao_informado_ori:,} ({nao_informado_ori/total*100:.1f}%)')
print()
print('Distribuição de gênero após normalização:')
df_silver.groupBy('genero').count().orderBy(F.desc('count')).show()
print('Distribuição de origem após normalização:')
df_silver.groupBy('origem').count().orderBy(F.desc('count')).show()
print('Distribuição de faixa_etaria:')
df_silver.groupBy('faixa_etaria').count().orderBy('faixa_etaria').show()

In [0]:
#Grava silver.dim_clientes
#overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.dim_clientes')
)

print(f'{silver_schema}.dim_clientes gravada com {df_silver.count():,} registros.')